In [510]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/mbti-tune/data")
PRETRAIN_DIR = BASE_DIR / "raw/pretrain"
MBTI_DIR = BASE_DIR / "raw/mbti_playlists"
RAW_PLAYLIST_DIR = BASE_DIR / "raw/raw_playlists"

PROCESSED_DIR = BASE_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [511]:
def folder_audit(path: Path):
    print(f"\n Folder: {path}")
    files = list(path.rglob("*"))

    csv_files = [f for f in files if f.suffix == ".csv"]

    print(f"Total items: {len(files)}")
    print(f"CSV files: {len(csv_files)}")

    for f in csv_files[:10]:
        try:
            print(f" - {f.name} | {pd.read_csv(f).shape}")
        except:
            print(f" - {f.name} | (failed to read)")

In [512]:
folder_audit(RAW_DIR / "pretrain")
folder_audit(RAW_DIR / "mbti_playlists")
folder_audit(RAW_DIR / "raw_playlists")


 Folder: /content/drive/MyDrive/mbti-tune/data/raw/pretrain
Total items: 1
CSV files: 1
 - spotify_tracks.csv | (114000, 21)

 Folder: /content/drive/MyDrive/mbti-tune/data/raw/mbti_playlists
Total items: 16
CSV files: 16
 - ESFJ.csv | (115, 49)
 - ENFJ.csv | (302, 49)
 - ENTJ.csv | (303, 49)
 - ESFP.csv | (231, 49)
 - ESTJ.csv | (122, 49)
 - ESTP.csv | (306, 49)
 - INFJ.csv | (304, 49)
 - INFP.csv | (295, 49)
 - INTJ.csv | (307, 49)
 - INTP.csv | (299, 49)

 Folder: /content/drive/MyDrive/mbti-tune/data/raw/raw_playlists
Total items: 342
CSV files: 326
 - ESFJ songs.csv | (50, 23)
 - ESFJ Kpop Songs.csv | (41, 23)
 - ESFJ.csv | (98, 23)
 - esfj(1).csv | (367, 23)
 - esfj(2).csv | (367, 23)
 - ESFJ(3).csv | (16, 23)
 - ESFJ.csv | (219, 23)
 - esfj(4).csv | (48, 23)
 - ESFJ MIX 2025（╹◡╹）.csv | (39, 23)
 - a playlist of ESFJ.csv | (7, 23)


In [513]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# PATHS
# =========================
spotify_path = PRETRAIN_DIR / "spotify_tracks.csv"
mbti_files = list(MBTI_DIR.rglob("*.csv"))
raw_files = list(RAW_PLAYLIST_DIR.rglob("*.csv"))

# =========================
# LOAD DATA
# =========================
spotify = pd.read_csv(spotify_path)
mbti = pd.concat([pd.read_csv(f) for f in mbti_files], ignore_index=True)

# IMPORTANT: attach MBTI from folder name
def extract_mbti(path):
    return Path(path).parent.name.upper()

raw_list = []
for f in raw_files:
    df = pd.read_csv(f)
    df["mbti"] = extract_mbti(f)
    raw_list.append(df)

raw = pd.concat(raw_list, ignore_index=True)

# Normalize columns for raw DataFrame
raw = normalize_columns(raw)

# =========================
# UNIVERSAL AUDIT FUNCTION
# =========================
def audit(df, name):
    print("\n" + "="*90)
    print(f"DATASET: {name}")
    print("="*90)

    print(f"Shape: {df.shape}")
    print(f"\nColumns ({len(df.columns)}):")
    print(df.columns.tolist())

    print("\nDtypes summary:")
    print(df.dtypes.value_counts())

    print("\nMissing (% top 15):")
    print((df.isna().mean() * 100).sort_values(ascending=False).head(15))

    print("\nDuplicate rows (full row duplicates):", df.duplicated().sum())

    print("\nNumeric columns:", df.select_dtypes(include=np.number).shape[1])

    print("\nObject columns:", df.select_dtypes(include="object").shape[1])

# =========================
# RUN AUDIT (BEFORE CLEANING)
# =========================
audit(spotify, "SPOTIFY RAW")
audit(mbti, "MBTI RAW")
audit(raw, "RAW PLAYLISTS RAW")

# =========================
# DEEP DUPLICATE CHECK (IMPORTANT FOR YOUR CASES)
# =========================

def duplicate_song_check(df, cols):
    print("\nDuplicate (subset):", cols)
    print(df.duplicated(subset=cols).sum())

# Spotify: track-level uniqueness
duplicate_song_check(spotify, ["track_id"])

# MBTI: playlist-level uniqueness
duplicate_song_check(mbti, ["playlist_id"])

# RAW: song-level (ignore MBTI later for grouping)
duplicate_song_check(raw, ["song", "artist", "bpm", "duration"])


DATASET: SPOTIFY RAW
Shape: (114000, 21)

Columns (21):
['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Dtypes summary:
float64    9
int64      6
object     5
bool       1
Name: count, dtype: int64

Missing (% top 15):
artists         0.000877
track_name      0.000877
album_name      0.000877
Unnamed: 0      0.000000
track_id        0.000000
popularity      0.000000
duration_ms     0.000000
explicit        0.000000
danceability    0.000000
energy          0.000000
key             0.000000
loudness        0.000000
mode            0.000000
speechiness     0.000000
acousticness    0.000000
dtype: float64

Duplicate rows (full row duplicates): 0

Numeric columns: 15

Object columns: 5

DATASET: MBTI RAW
Shape: (4201, 49)

Columns (49):
['mbti', 'function_pair', 'p

In [514]:
def audit(df, name):
    print("\n" + "="*90)
    print(f"{name}")
    print("="*90)

    print(f"Shape: {df.shape}")
    print(f"Columns: {len(df.columns)}")

    print("\nDtypes:")
    print(df.dtypes.value_counts())

    print("\nMissing % (top 10):")
    print((df.isnull().mean()*100).sort_values(ascending=False).head(10))

    print("\nDuplicate rows:", df.duplicated().sum())

    num_cols = df.select_dtypes(include=np.number).shape[1]
    print("\nNumeric columns:", num_cols)

In [521]:
spotify_path = PRETRAIN_DIR / "spotify_tracks.csv"
spotify = pd.read_csv(spotify_path)

# CLEANING
spotify = spotify.drop(columns=[c for c in spotify.columns if "Unnamed" in c], errors="ignore")

spotify[["artists","album_name","track_name"]] = spotify[
    ["artists","album_name","track_name"]
].fillna("unknown")


# enforce dtypes
num_cols = spotify.select_dtypes(include=["int64","float64"]).columns
spotify[num_cols] = spotify[num_cols].apply(pd.to_numeric, errors="coerce")

# boolean fix
if "explicit" in spotify.columns:
    spotify["explicit"] = spotify["explicit"].astype(bool)

# drop missing critical rows (very few anyway)
spotify = spotify.dropna(subset=["track_id"])

# AUDIT
audit(spotify, "SPOTIFY CLEAN (AUTOENCODER)")

# SAVE
spotify.to_csv(PROCESSED_DIR / "spotify_clean.csv", index=False)
print("\nSaved:", PROCESSED_DIR / "spotify_clean.csv")


SPOTIFY CLEAN (AUTOENCODER)
Shape: (114000, 20)
Columns: 20

Dtypes:
float64    9
object     5
int64      5
bool       1
Name: count, dtype: int64

Missing % (top 10):
track_id        0.0
artists         0.0
album_name      0.0
track_name      0.0
popularity      0.0
duration_ms     0.0
explicit        0.0
danceability    0.0
energy          0.0
key             0.0
dtype: float64

Duplicate rows: 450

Numeric columns: 14

Saved: /content/drive/MyDrive/mbti-tune/data/processed/spotify_clean.csv


In [516]:
mbti_files = list(MBTI_DIR.rglob("*.csv"))
mbti = pd.concat([pd.read_csv(f) for f in mbti_files], ignore_index=True)

# CLEAN COLUMN NAMES
mbti.columns = (
    mbti.columns
    .str.lower()
    .str.replace("#", "sharp")
    .str.replace("/", "_")
)

# CLEAN MBTI LABEL
if "mbti" in mbti.columns:
    mbti["mbti"] = mbti["mbti"].astype(str).str.upper()

# FIX NUMERIC TYPES
for c in mbti.columns:
    if "count" in c or "mean" in c or "stdev" in c:
        mbti[c] = pd.to_numeric(mbti[c], errors="coerce")

# AUDIT
audit(mbti, "MBTI CLEAN (CLASSIFIER)")

# SAVE
mbti.to_csv(PROCESSED_DIR / "mbti_clean.csv", index=False)
print("\nSaved:", PROCESSED_DIR / "mbti_clean.csv")


MBTI CLEAN (CLASSIFIER)
Shape: (4201, 49)
Columns: 49

Dtypes:
float64    44
object      4
int64       1
Name: count, dtype: int64

Missing % (top 10):
dsharp_ebminor_count    0.095215
csharp_dbminor_count    0.047608
gsharp_abminor_count    0.047608
function_pair           0.000000
track_count             0.000000
danceability_mean       0.000000
playlist_name           0.000000
playlist_id             0.000000
energy_stdev            0.000000
loudness_mean           0.000000
dtype: float64

Duplicate rows: 73

Numeric columns: 45

Saved: /content/drive/MyDrive/mbti-tune/data/processed/mbti_clean.csv


In [517]:
raw_files = list(RAW_PLAYLIST_DIR.rglob("*.csv"))

dfs = []

for f in raw_files:
    df = pd.read_csv(f)

    # ADD MBTI FROM FOLDER NAME
    mbti_label = f.parent.name.upper()
    df["mbti"] = mbti_label

    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)

# CLEAN COLUMN NAMES
raw.columns = (
    raw.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("[()#]", "", regex=True)
)

# unify naming issues
raw = raw.rename(columns={
    "loud_(db)": "loud_db",
    "spotify_track_id": "track_id"
})

# TYPE FIXING
num_guess = ["bpm","energy","dance","acoustic","instrumental",
             "valence","speech","live","loud_db","popularity",
             "duration","key","time_signature"]

for c in num_guess:
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

identity_cols = ["mbti"]

for c in ["track_id", "song", "artist", "bpm", "duration"]:
    if c in raw.columns:
        identity_cols.append(c)

raw = raw.drop(columns=["duration", "key"], errors="ignore")

raw = raw.drop_duplicates(
    subset=["mbti", "song", "artist", "bpm", "popularity"]
)

# AUDIT
audit(raw, "RAW CLEAN (MBTI + AUTOENCODER)")

# SAVE FULL VERSION
raw.to_csv(PROCESSED_DIR / "raw_playlists_clean.csv", index=False)

print("\nSaved:", PROCESSED_DIR / "raw_playlists_clean.csv")

raw["mbti"].value_counts()


RAW CLEAN (MBTI + AUTOENCODER)
Shape: (45902, 22)
Columns: 22

Dtypes:
object    11
int64     11
Name: count, dtype: int64

Missing % (top 10):
genres        38.420984
isrc           0.041393
album          0.034857
song           0.032678
artist         0.032678
               0.000000
energy         0.000000
camelot        0.000000
bpm            0.000000
popularity     0.000000
dtype: float64

Duplicate rows: 0

Numeric columns: 11

Saved: /content/drive/MyDrive/mbti-tune/data/processed/raw_playlists_clean.csv


,count
mbti,
ISFP,6158
INFP,4968
ISFJ,3785
INFJ,3211
ENFP,3201
ISTP,2969
INTP,2773
INTJ,2641
ESFP,2546


In [518]:
summary = pd.DataFrame({
    "dataset": ["spotify", "mbti", "raw"],
    "rows": [len(spotify), len(mbti), len(raw)],
    "cols": [spotify.shape[1], mbti.shape[1], raw.shape[1]],
    "numeric_cols": [
        spotify.select_dtypes(include=np.number).shape[1],
        mbti.select_dtypes(include=np.number).shape[1],
        raw.select_dtypes(include=np.number).shape[1],
    ],
    "object_cols": [
        spotify.select_dtypes(include="object").shape[1],
        mbti.select_dtypes(include="object").shape[1],
        raw.select_dtypes(include="object").shape[1],
    ],
    "duplicates": [
        spotify.duplicated().sum(),
        mbti.duplicated().sum(),
        raw.duplicated().sum(),
    ]
})

summary

,dataset,rows,cols,numeric_cols,object_cols,duplicates
0,spotify,114000,20,14,5,450
1,mbti,4201,49,45,4,73
2,raw,45902,22,11,11,0


In [519]:
import pandas as pd
import numpy as np

def full_audit(df, name, show_mbti_dist=False):
    print("\n" + "="*90)
    print(f"DATASET: {name}")
    print("="*90)

    # ---------------- SHAPE ----------------
    print(f"Shape: {df.shape}  (rows={df.shape[0]}, cols={df.shape[1]})")

    # ---------------- COLUMNS ----------------
    print(f"\nColumns ({len(df.columns)}):")
    print(df.columns.tolist())

    # ---------------- DTYPE SUMMARY ----------------
    print("\nDtype Summary:")
    print(df.dtypes.value_counts())

    print("\nFull Dtypes:")
    print(df.dtypes)

    # ---------------- MISSING ----------------
    print("\nMissing % (top 15):")
    missing = (df.isnull().mean() * 100).sort_values(ascending=False)
    print(missing.head(15))

    # ---------------- DUPLICATES ----------------
    print("\nDuplicate rows (full row):", df.duplicated().sum())

    # ---------------- COLUMN TYPES ----------------
    num_cols = df.select_dtypes(include=np.number).shape[1]
    obj_cols = df.select_dtypes(include="object").shape[1]

    print("\nNumeric columns:", num_cols)
    print("Object columns:", obj_cols)

    # ---------------- SAMPLE ----------------
    print("\nSample columns:")
    print(df.columns[:15].tolist(), "...")

    # ---------------- MBTI DISTRIBUTION ----------------
    if show_mbti_dist and "mbti" in df.columns:
        print("\nMBTI distribution:")
        print(df["mbti"].value_counts())

    print("\n" + "-"*90)

In [520]:
full_audit(spotify, "SPOTIFY CLEAN (AUTOENCODER)")
full_audit(mbti, "MBTI CLEAN (CLASSIFIER)", show_mbti_dist=True)
full_audit(raw, "RAW CLEAN (MBTI + AUTOENCODER)")


DATASET: SPOTIFY CLEAN (AUTOENCODER)
Shape: (114000, 20)  (rows=114000, cols=20)

Columns (20):
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Dtype Summary:
float64    9
object     5
int64      5
bool       1
Name: count, dtype: int64

Full Dtypes:
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
t